# Reproducing the Geometry of Truth with **murano**

> *The Geometry of Truth: Emergent Linear Structure in LLM Representations of
> True/False Datasets.* Samuel Marks, Max Tegmark. arXiv 2023.
> [arXiv:2310.06824](https://arxiv.org/abs/2310.06824)

This tutorial reproduces the paper's central claim with murano: a language model
represents the truth of a factual statement **linearly**. A single direction separates true and false claims inside the residual stream. This same direction is shared across unrelated topics, and adding or subtracting it in the residual stream causally flips the model's judgement of whether a statement is true.

**What you will learn.**
1. Record last-token residual activations over true/false statements with `Record`.
2. See the true/false split in the top principal components, and watch it emerge
   across layers.
3. Fit logistic (LR) and mass-mean (MM) probes on one topic and measure how they
   transfer to unrelated topics, where affirmative and negated statements
   anti-correlate.
4. Build the mass-mean truth direction with `SteeringVector` and use
   `forward_logits` to add or subtract it, measuring how strongly it steers the
   model's truth judgement.

**The data.** The paper's own curated datasets of simple factual statements —
city locations, size comparisons, Spanish–English translations, and their
negations — each statement labelled true or false.

**On fidelity.** This follows the paper's method: last-token residual reads at the
probe layer, PCA via SVD, the mass-mean and logistic probes with the paper's
whitened decision rule, and the mass-mean direction applied unscaled over an
intervention band, scored by the paper's normalized indirect effect. Where the
paper reports a number for this model, it is shown side by side.

**The result you will find.** True and false statements separate along the first
principal component, and the split sharpens through the early-middle layers. A
probe trained on one topic transfers to unrelated topics, while negated datasets
anti-correlate. Adding the truth direction pushes false statements toward TRUE and
subtracting it pushes true statements toward FALSE, recovering most of the natural
true/false gap.

**Requirements.** A GPU with room for LLaMA-2-13B (about 26 GB in fp16; the
checkpoint is gated, so accept the license and `huggingface-cli login` first),
`murano-interp[probe,plot]`, and network access to pull the model and datasets on
first run. Run the cells top to bottom.

## 1. Setup

Load a model. The truth structure the paper studies is a property of a pretrained
(non-instruction-tuned) causal LM, so this uses base LLaMA-2-13B, the paper's
mid-scale model. Install `murano-interp[probe,plot]`, enable the Apple-Silicon CPU
fallback, and clone the paper's datasets.

In [1]:
# Install deps, enable Apple-Silicon fallback, clone datasets, make output dirs.
import os, subprocess, sys

# Let ops torch/nnterp haven't implemented for the Metal (MPS) backend fall back
# to CPU instead of raising. Must precede the first MPS op; harmless on CUDA/CPU.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")


def ensure(pkg, spec=None, fatal=True):
    """Import `pkg`, pip-installing `spec` if missing. Non-fatal installs just warn
    (e.g. in a uv-managed venv without pip) so the notebook still runs."""
    try:
        __import__(pkg)
        return
    except ImportError:
        pass
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec or pkg], check=True)
    except Exception as e:
        msg = f"could not install {pkg} ({e})."
        if fatal:
            raise RuntimeError(msg + " Install it manually and re-run.") from e
        print(f"warning: {msg} continuing without it.")


ensure("murano", "murano-interp[probe,plot]")
ensure("nbformat")              # required for inline plotly rendering
ensure("kaleido")              # required for figure -> PDF export
ensure("ipywidgets", fatal=False)  # nicer tqdm bars; optional

GOT_DIR = "geometry-of-truth"
if not os.path.isdir(GOT_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/saprmarks/geometry-of-truth.git", GOT_DIR], check=True)

PLOTS_DIR, TABLES_DIR = "plots", "tables"
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
print("datasets:", GOT_DIR, "| figures ->", PLOTS_DIR, "| tables ->", TABLES_DIR)

datasets: geometry-of-truth | figures -> plots | tables -> tables


In [2]:
import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from murano import MuranoModel, Pipeline
from murano.dataset import LabeledDataset, MuranoDataset
from murano.steps.load import Load
from murano.steps.record import Record
from murano.steps.train import SteeringVector

torch.set_grad_enabled(False)  # extraction and probing are gradient-free

# --- Device: CUDA > Apple-Silicon MPS > CPU ---------------------------------
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# dtype: fp16 on CUDA; bfloat16 elsewhere. LLaMA-2-13B is ~26 GB in 16-bit but
# ~52 GB in fp32, so we never keep the weights in fp32 (downstream math upcasts via
# .float() anyway). At 26 GB the weights fit a 40 GB A100 but not a typical laptop
# GPU / MPS buffer -- expect the CPU fallback below off a datacenter GPU.
dtype = torch.float16 if DEVICE == "cuda" else torch.bfloat16

# --- Model: base (pretrained) LLaMA-2-13B, the paper's mid-scale model ---------
# The paper runs BASE LLaMA-2, per the original config.ini (`-hf`, not `-chat-hf`),
# with no instruction/RLHF tuning. The checkpoint is gated: accept the license and
# `huggingface-cli login` first.
MODEL_ID = "meta-llama/Llama-2-13b-hf"

SUBSAMPLE = None  # full datasets (methodologically identical to the original)


def load_model(model_ref):
    """Load on the best available accelerator, falling back to CPU if it can't fit
    the model. Apple-Silicon MPS caps single-buffer allocations well below RAM, so
    a 13B model overflows the GPU ('Invalid buffer size') and must run on CPU."""
    device_map = "mps" if DEVICE == "mps" else "auto"
    try:
        return MuranoModel(model_ref, device_map=device_map, dtype=dtype), DEVICE
    except Exception as e:
        if DEVICE == "cpu":
            raise
        print(f"warning: could not load on {DEVICE} ({e});\n  falling back to CPU (slower but fits in RAM).")
        return MuranoModel(model_ref, device_map="cpu", dtype=dtype), "cpu"


model, DEVICE = load_model(MODEL_ID)
BATCH = 8  # 13B is large; keep batches small

# Layer settings for LLaMA-2-13B, taken verbatim from the paper's config.ini:
#   PROBE_LAYER      -- residual-stream layer read for PCA, probing, and the direction.
#   INTERVENE_LAYERS -- the causal-intervention band, intervene_layer..probe_layer.
PROBE_LAYER = 14
INTERVENE_LAYERS = list(range(8, PROBE_LAYER + 1))

# Layers at which to show the truth direction emerging across depth.
EMERGENCE_LAYERS = sorted({int(round(x)) for x in np.linspace(2, model.n_layers - 2, 5)})

print(f"{MODEL_ID}: {model.n_layers} layers, d_model={model.d_model}, device={DEVICE}, dtype={dtype}")
print(f"probe layer={PROBE_LAYER} | intervene={INTERVENE_LAYERS} | emergence={EMERGENCE_LAYERS}")


def get_pcs(X, k=2):
    """Top-k principal components of X (the paper's utils.get_pcs, via SVD)."""
    X = X - X.mean(0)
    _, _, V = torch.linalg.svd(X, full_matrices=False)
    return V[:k].T  # [d, k]


def save_pdf(fig, name):
    """Save a plotly figure to plots/<name>.pdf (best-effort) and return it for
    inline display."""
    path = os.path.join(PLOTS_DIR, name)
    try:
        fig.write_image(path)
        print("saved", path)
    except Exception as e:
        print(f"warning: could not write {path} ({e}). Is kaleido installed?")
    return fig


def write_latex(latex, name):
    """Write a LaTeX table string to tables/<name> and echo it."""
    path = os.path.join(TABLES_DIR, name)
    with open(path, "w") as f:
        f.write(latex)
    print("saved", path, "\n")
    print(latex)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

meta-llama/Llama-2-13b-hf: 40 layers, d_model=5120, device=cuda, dtype=torch.float16
probe layer=14 | intervene=[8, 9, 10, 11, 12, 13, 14] | emergence=[2, 11, 20, 29, 38]


## 2. Data and activation recording

The paper's own curated datasets of true/false statements, pulled from the
original repo. `Record(position="last")` captures the residual stream at the final
token (the period) of each statement — the same site the paper reads
(`layers[l].output[0][:,-1,:]`). We record only the layers each step needs, to
stay light on memory.

In [3]:
import hashlib

CORE = ["cities", "neg_cities", "larger_than", "smaller_than", "sp_en_trans", "neg_sp_en_trans"]


def _seed(name):
    """Stable per-dataset seed (independent of PYTHONHASHSEED)."""
    return int(hashlib.md5(name.encode()).hexdigest()[:8], 16)


def load_tf(name, subsample=SUBSAMPLE):
    """Return (true_statements, false_statements) for dataset `name`.
    Deterministic per name so every section (and every session) sees the same
    subsample."""
    df = pd.read_csv(f"{GOT_DIR}/datasets/{name}.csv")
    if subsample and len(df) > subsample:
        idx = np.random.default_rng(_seed(name)).permutation(len(df))[:subsample]
        df = df.iloc[idx]
    true = df[df.label == 1]["statement"].tolist()
    false = df[df.label == 0]["statement"].tolist()
    return true, false


_store_cache = {}


def record_at(name, layers):
    """Record `name` at `layers` as a labelled activation store (cached)."""
    key = (name, tuple(layers))
    if key not in _store_cache:
        true, false = load_tf(name)
        _store_cache[key] = Pipeline([
            Load(LabeledDataset(texts=true + false, labels=[1] * len(true) + [0] * len(false))),
            Record(model, layers=list(layers), position="last", batch_size=BATCH),
        ]).run()["record"]
    return _store_cache[key]


for name in CORE:
    t_, f_ = load_tf(name)
    print(f"{name:16s} {len(t_):4d} true / {len(f_):4d} false")
t0, f0 = load_tf("cities")
print("\nexample true :", t0[0])
print("example false:", f0[0])

cities            748 true /  748 false
neg_cities        748 true /  748 false
larger_than       990 true /  990 false
smaller_than      990 true /  990 false
sp_en_trans       177 true /  177 false
neg_sp_en_trans   177 true /  177 false

example true : The city of Krasnodar is in Russia.
example false: The city of Krasnodar is in South Africa.


## 3. Linear structure: true vs. false in the principal components

Read the residual stream at the probe layer over each dataset and project onto its
top two principal components — an unsupervised view, since the labels never enter
the axes. If truth is encoded linearly, true and false statements separate along a
principal component. We then contrast an affirmative dataset with its negation
(`cities` vs. `neg_cities`): the two truth directions come out near-orthogonal, the
first sign that negation is represented separately.

In [12]:
PANEL_SETS = ["cities", "sp_en_trans", "larger_than"]
probe_stores = {name: record_at(name, [PROBE_LAYER]) for name in CORE}


def pca_frame(store, layer):
    X = store.activations[(layer, "residual")].float().cpu()
    X = X - X.mean(0)
    proj = X @ get_pcs(X, k=2)
    y = store.labels.float().cpu()
    return pd.DataFrame({"PC1": proj[:, 0], "PC2": proj[:, 1],
                         "truth": ["true" if v == 1 else "false" for v in y]})


fig = make_subplots(rows=1, cols=len(PANEL_SETS), subplot_titles=PANEL_SETS)
for j, name in enumerate(PANEL_SETS, start=1):
    fr = pca_frame(probe_stores[name], PROBE_LAYER).sample(frac=1, random_state=0)
    for truth, colr in [("true", "#3b6bd6"), ("false", "#d64545")]:
        sub = fr[fr.truth == truth]
        fig.add_trace(go.Scatter(x=sub.PC1, y=sub.PC2, mode="markers", name=truth,
                                 marker=dict(color=colr, size=5), showlegend=(j == 1)),
                      row=1, col=j)
# No title; legend overlaid in the bottom-right of the rightmost panel.
fig.update_layout(width=920, height=300, margin=dict(l=30, r=20, t=30, b=30),
                  legend=dict(x=1, y=0, xanchor="right", yanchor="bottom", font=dict(size=16),
                              bgcolor="rgba(255,255,255,0.65)", borderwidth=0))
fig.update_annotations(font_size=20)  # larger dataset (subplot) titles
save_pdf(fig, "fig1_pca_separation.pdf")
fig

/scratch/409116/ipykernel_2893094/3384451645.py:81: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(path)


saved plots/fig1_pca_separation.pdf


In [5]:
# cities vs neg_cities — the two truth directions are near-orthogonal.
neg_store = probe_stores["neg_cities"]
pos_store = probe_stores["cities"]


def four_class_frame():
    rows = []
    for store, ds in [(pos_store, "cities"), (neg_store, "neg_cities")]:
        X = store.activations[(PROBE_LAYER, "residual")].float().cpu()
        X = X - X.mean(0)
        proj = X @ get_pcs(X, k=2)  # per-dataset PCs, as the paper plots each pair
        y = store.labels.float().cpu()
        for i in range(len(y)):
            rows.append(dict(PC1=float(proj[i, 0]), PC2=float(proj[i, 1]),
                             cls=f"{ds}: {'true' if y[i]==1 else 'false'}"))
    return pd.DataFrame(rows)


fr = four_class_frame().sample(frac=1, random_state=0)
cmap = {"cities: true": "#3b6bd6", "cities: false": "#d64545",
        "neg_cities: true": "#e8b800", "neg_cities: false": "#7a4fd6"}
fig = px.scatter(fr, x="PC1", y="PC2", color="cls", color_discrete_map=cmap,
                 title="cities vs neg_cities")
# Centered title; legend overlaid in the top-right of the figure.
fig.update_layout(width=620, height=460, legend_title_text="",
                  title=dict(x=0.5, xanchor="center"),
                  margin=dict(l=30, r=20, t=40, b=30),
                  legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top",
                              bgcolor="rgba(255,255,255,0.65)", borderwidth=0))
save_pdf(fig, "fig3c_negation.pdf")
fig

/scratch/409116/ipykernel_2893094/3384451645.py:81: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(path)


saved plots/fig3c_negation.pdf


## 4. The truth direction emerges across layers

The linear structure is not present at the input and fully formed at the output —
it develops with depth. Sweep the same PCA across a range of layers: the true/false
split is absent early and sharpens through the early-middle layers, where the truth
direction takes shape.

In [6]:
emb = record_at("cities", EMERGENCE_LAYERS)
fig = make_subplots(rows=1, cols=len(EMERGENCE_LAYERS),
                    subplot_titles=[f"layer {l}" for l in EMERGENCE_LAYERS])
for j, layer in enumerate(EMERGENCE_LAYERS, start=1):
    fr = pca_frame(emb, layer).sample(frac=1, random_state=0)
    for truth, colr in [("true", "#3b6bd6"), ("false", "#d64545")]:
        sub = fr[fr.truth == truth]
        fig.add_trace(go.Scatter(x=sub.PC1, y=sub.PC2, mode="markers", name=truth,
                                 marker=dict(color=colr, size=4), showlegend=(j == 1)),
                      row=1, col=j)
# No title; keep only the per-layer subtitles, legend overlaid bottom-right.
fig.update_layout(width=1050, height=220, margin=dict(l=20, r=20, t=26, b=20),
                  legend=dict(x=1, y=0, xanchor="right", yanchor="bottom",
                              bgcolor="rgba(255,255,255,0.65)", borderwidth=0))
fig.update_xaxes(showticklabels=False); fig.update_yaxes(showticklabels=False)
save_pdf(fig, "fig7_pca_emergence.pdf")
fig

/scratch/409116/ipykernel_2893094/3384451645.py:81: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(path)


saved plots/fig7_pca_emergence.pdf


## 5. Generalization across topics

If the model holds one truth direction rather than a separate feature per topic, a
probe trained on one topic should read truth on unrelated topics. Following the
paper's protocol, we mean-center every dataset, fit a logistic (LR) and a mass-mean
(MM) probe on two training medleys (`cities+neg_cities` and
`larger_than+smaller_than`), and score each dataset — held-out datasets on a
validation split the probe never saw, every other dataset on its full set. The MM
probe uses the paper's whitened decision rule in-distribution and the raw direction
out-of-distribution; LR is plain logistic regression. Probes transfer across
topics, and affirmative and negated datasets anti-correlate.

In [16]:
from sklearn.linear_model import LogisticRegression

MEDLEYS = {"cities+neg_cities": ["cities", "neg_cities"],
           "larger_than+smaller_than": ["larger_than", "smaller_than"]}
COLS = [("LR", "cities+neg_cities"), ("LR", "larger_than+smaller_than"),
        ("MM", "cities+neg_cities"), ("MM", "larger_than+smaller_than")]

# Paper's generalization protocol (generalization.ipynb): split the two training-
# medley datasets 80/20, fit probes on the pooled 80% train half, then score each
# held-out (in-medley) dataset on its 20% val split and every other dataset on its
# full set. Fixed seed for reproducibility (the paper draws a random one).
GEN_SPLIT, GEN_SEED = 0.8, 0


def _centered(name):
    """Per-dataset mean-centered activations + labels at the probe layer
    (paper's collect_acts(center=True))."""
    st = probe_stores[name]
    X = st.activations[(PROBE_LAYER, "residual")].float().cpu()
    y = st.labels.float().cpu()
    return X - X.mean(0), y


def _train_mask(n):
    """Random train mask of floor(GEN_SPLIT*n) rows (paper's randperm(n) < k)."""
    g = torch.Generator().manual_seed(GEN_SEED)
    return torch.randperm(n, generator=g) < int(GEN_SPLIT * n)


def _mm_probe(X, y, atol=1e-3):
    """Mass-mean probe (paper's MMProbe.from_data): truth direction plus the
    pseudo-inverse within-class covariance used by the whitened (iid) rule."""
    pos, neg = X[y == 1], X[y == 0]
    direction = pos.mean(0) - neg.mean(0)
    centered = torch.cat([pos - pos.mean(0), neg - neg.mean(0)], 0)
    cov = centered.t() @ centered / X.shape[0]
    inv = torch.linalg.pinv(cov, hermitian=True, atol=atol)  # ~5120x5120, CPU: a few s
    return direction, inv


def build_grid():
    """Reproduce the paper's generalization grid. MM uses the whitened decision
    rule (iid=True, sigmoid(x . inv . dir)) for in-medley val and the raw direction
    (iid=False) out-of-distribution; LR is plain logistic regression either way."""
    grid = {}
    for medley, members in MEDLEYS.items():
        train_X, train_y, val = [], [], {}
        for d in members:
            X, y = _centered(d)
            m = _train_mask(len(y))
            train_X.append(X[m]); train_y.append(y[m])
            val[d] = (X[~m], y[~m])
        Xtr, ytr = torch.cat(train_X), torch.cat(train_y)

        lr = LogisticRegression(max_iter=1000).fit(Xtr.numpy(), ytr.numpy())
        mm_dir, mm_inv = _mm_probe(Xtr, ytr)

        for probe in ("LR", "MM"):
            grid[(probe, medley)] = {}
            for test in CORE:
                Xte, yte = val[test] if test in members else _centered(test)
                if probe == "LR":
                    pred = torch.as_tensor(lr.predict(Xte.numpy()), dtype=torch.float)
                else:  # in-medley -> whitened (iid) rule; OOD -> raw direction
                    score = Xte @ mm_inv @ mm_dir if test in members else Xte @ mm_dir
                    pred = (score > 0).float()
                grid[(probe, medley)][test] = 100 * (pred == yte).float().mean().item()
    return grid


GRID = build_grid()
# Reproduced grid (rows = test set, cols = probe/train-medley).
REPRO = np.array([[GRID[(p, tr)][test] for (p, tr) in COLS] for test in CORE])

# Paper's exact LLaMA-2-13B values, transcribed from the paper's 13B generalization
# grid. Columns pulled per (probe, train-medley):
#   [LR / cities+neg_cities, LR / larger_than+smaller_than,
#    MM / cities+neg_cities, MM / larger_than+smaller_than].
PAPER = {  # test set -> [LR/cn, LR/ls, MM/cn, MM/ls]
    "cities":          [100, 84, 100, 89],
    "neg_cities":      [100, 99, 100, 97],
    "larger_than":     [ 88, 100, 93, 100],
    "smaller_than":    [ 86, 100, 86, 100],
    "sp_en_trans":     [100, 95, 98, 97],
    "neg_sp_en_trans": [ 94, 81, 96, 85],
}
PAPER_M = np.array([PAPER[t] for t in CORE], dtype=float)

# Compact side-by-side heatmaps. Columns 0-1 are LR, columns 2-3 are MM, so the
# probe name is written once (centered over its two columns) and the x tick shows
# only the train medley. Dataset names appear once, on the left panel's rows.
xpos = list(range(len(COLS)))
col_meds = [tr.replace("+", "+<br>") for (_, tr) in COLS]  # train medley per column
panel_titles = ["Marks & Tegmark (2023)", "Murano reproduction"]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.03)
for c, M in enumerate([PAPER_M, REPRO], start=1):
    sfx = "" if c == 1 else "2"
    fig.add_trace(go.Heatmap(z=M, x=xpos, y=CORE, zmin=0, zmax=100,
                             colorscale="Blues", showscale=(c == 2),
                             colorbar=dict(title="acc %")), row=1, col=c)
    for i in range(len(CORE)):
        for j in range(len(COLS)):
            fig.add_annotation(x=xpos[j], y=CORE[i], text=f"{M[i, j]:.0f}",
                               showarrow=False, row=1, col=c,
                               font=dict(size=13, color="white" if M[i, j] >= 55 else "#222"))
    # Panel title and the LR / MM group headers, centered over their two columns.
    fig.add_annotation(x=0.5, y=1.22, xref=f"x{sfx} domain", yref=f"y{sfx} domain",
                       text=f"{panel_titles[c - 1]}", showarrow=False,
                       font=dict(size=20))
    for label, xc in [("LR", 0.5), ("MM", 2.5)]:
        fig.add_annotation(x=xc, y=1.1, xref=f"x{sfx}", yref=f"y{sfx} domain",
                           text=f"<b>{label}</b>", showarrow=False, font=dict(size=17))

fig.update_layout(width=860, height=470, margin=dict(l=115, r=70, t=84, b=96))
fig.update_xaxes(tickvals=xpos, ticktext=col_meds, tickfont=dict(size=11))
fig.update_yaxes(autorange="reversed", tickfont=dict(size=13))
fig.update_yaxes(showticklabels=False, row=1, col=2)  # dataset names once, on the left
save_pdf(fig, "fig10_generalization.pdf")
fig

/scratch/409116/ipykernel_2893094/3384451645.py:81: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(path)


saved plots/fig10_generalization.pdf


In [8]:
# LaTeX table: generalization for the cities+neg_cities medley (paper's headline),
# incl. the Fig. 5a "average over held-out test sets" row.  -> tables/generalization.txt
# Paper column uses the LLaMA-2-13B grid (Figure 10, Appendix D) via the PAPER dict.
HELDOUT = [d for d in CORE if d not in MEDLEYS["cities+neg_cities"]]
idx = {t: i for i, t in enumerate(CORE)}
paper_lr = {t: PAPER[t][0] for t in CORE}
paper_mm = {t: PAPER[t][2] for t in CORE}
repro_lr = {t: REPRO[idx[t], 0] for t in CORE}
repro_mm = {t: REPRO[idx[t], 2] for t in CORE}


def esc(s):
    return s.replace("_", r"\_")


rows = []
for t in CORE:
    rows.append(f"    {esc(t):22s} & {paper_lr[t]:.0f} & {paper_mm[t]:.0f} & "
                f"{repro_lr[t]:.0f} & {repro_mm[t]:.0f} \\\\")
avg_p_lr = np.mean([paper_lr[t] for t in HELDOUT])
avg_p_mm = np.mean([paper_mm[t] for t in HELDOUT])
avg_r_lr = np.mean([repro_lr[t] for t in HELDOUT])
avg_r_mm = np.mean([repro_mm[t] for t in HELDOUT])
body = "\n".join(rows)
latex = rf"""\begin{{table}}[tbp]
  \centering\small
  \begin{{tabular}}{{lcccc}}
    \hline
     & \multicolumn{{2}}{{c}}{{\textbf{{\citet{{marks2023geometry}}}}}} & \multicolumn{{2}}{{c}}{{\textbf{{Murano}}}} \\
    Test set (train: cities+neg\_cities) & LR & MM & LR & MM \\
    \hline
{body}
    \hline
    Avg.\ (held-out, Fig.~5a) & {avg_p_lr:.0f} & {avg_p_mm:.0f} & {avg_r_lr:.0f} & {avg_r_mm:.0f} \\
    \hline
  \end{{tabular}}
  \caption{{Generalization accuracy (\%) of logistic (LR) and mass-mean (MM) probes
  trained on \texttt{{cities+neg\_cities}} at layer {PROBE_LAYER} of LLaMA-2-13B and
  tested on each dataset. Original values transcribed from \citet{{marks2023geometry}}
  Figure~10 (Appendix~D, 13B generalization grid); the held-out average is over the
  four held-out core datasets shown here (the paper's Figure~5a averages over all
  eleven evaluation datasets). Negated datasets (\texttt{{neg\_*}}) transfer poorly /
  anti-correlate, as in the paper.}}
  \label{{tab:got-generalization}}
\end{{table}}
"""
write_latex(latex, "generalization.txt")

saved tables/generalization.txt 

\begin{table}[tbp]
  \centering\small
  \begin{tabular}{lcccc}
    \hline
     & \multicolumn{2}{c}{\textbf{\citet{marks2023geometry}}} & \multicolumn{2}{c}{\textbf{Murano}} \\
    Test set (train: cities+neg\_cities) & LR & MM & LR & MM \\
    \hline
    cities                 & 100 & 100 & 100 & 100 \\
    neg\_cities            & 100 & 100 & 100 & 100 \\
    larger\_than           & 88 & 93 & 93 & 92 \\
    smaller\_than          & 86 & 86 & 76 & 87 \\
    sp\_en\_trans          & 100 & 98 & 99 & 98 \\
    neg\_sp\_en\_trans     & 94 & 96 & 97 & 96 \\
    \hline
    Avg.\ (held-out, Fig.~5a) & 92 & 93 & 91 & 93 \\
    \hline
  \end{tabular}
  \caption{Generalization accuracy (\%) of logistic (LR) and mass-mean (MM) probes
  trained on \texttt{cities+neg\_cities} at layer 14 of LLaMA-2-13B and
  tested on each dataset. Original values transcribed from \citet{marks2023geometry}
  Figure~10 (Appendix~D, 13B generalization grid); the held-out average is

## 6. Steering the model's truth judgement

The truth direction is not just decodable, it is causal: adding or subtracting it
in the residual stream flips whether the model judges a statement true or false. We
build the mass-mean direction on `cities+neg_cities` (murano's
`SteeringVector(normalize=False)`), then apply it over an intervention band while
the model judges `sp_en_trans` statements under a few-shot prompt, and quantify the
effect with the paper's **normalized indirect effect (NIE)**.

Splitting the eval set by ground-truth label gives four probability-difference
terms — `PD⁺` / `PD⁻`, the mean `P(TRUE) − P(FALSE)` over true / false statements
with no intervention; `PD*⁻`, false statements with the direction **added**; and
`PD*⁺`, true statements with the direction **subtracted** — from which the NIE
follows:

$$\text{NIE}_{f\to t}=\frac{PD^{*-}-PD^-}{PD^+-PD^-},\qquad
  \text{NIE}_{t\to f}=\frac{PD^{*+}-PD^+}{PD^--PD^+}.$$

An NIE of 1 means the intervention flips the judgement as confidently as a genuine
label; 0 means no effect.

In [9]:
# --- Direction: mass-mean on cities+neg_cities at the probe layer -------------
# SteeringVector(normalize=False) returns the raw mass-mean difference
# (true_mean - false_mean). This equals the paper's *rescaled* MM probe direction:
# interventions.py takes the unit probe direction and multiplies it by its own
# projection onto (true_mean - false_mean), which for the mass-mean probe returns
# exactly (true_mean - false_mean).
t_c, f_c = load_tf("cities")
t_n, f_n = load_tf("neg_cities")
direction = (Pipeline([
    Load(MuranoDataset(positive_texts=t_c + t_n, negative_texts=f_c + f_n)),
    Record(model, layers=[PROBE_LAYER], position="last", batch_size=BATCH),
    SteeringVector(normalize=False),
]).run()["steering"].direction_per_layer[(PROBE_LAYER, "residual")].to(DEVICE, dtype))

# Paper's 4-shot prompt for sp_en_trans (interventions.py, llama-2-13b entry).
prompt = ("The Spanish word 'jirafa' means 'giraffe'. This statement is: TRUE\n"
          "The Spanish word 'escribir' means 'to write'. This statement is: TRUE\n"
          "The Spanish word 'gato' means 'cat'. This statement is: TRUE\n"
          "The Spanish word 'aire' means 'silver'. This statement is: FALSE\n")
true_id = model.tokenizer.encode(" TRUE")[-1]
false_id = model.tokenizer.encode(" FALSE")[-1]
LEN_SUFFIX = len(model.tokenizer.encode("This statement is:"))
model.tokenizer.padding_side = "left"  # keep the suffix at the sequence end under padding

# Eval set: sp_en_trans, split by ground-truth label (the paper intervenes on the
# true and false subsets separately). Few-shot statements are excluded, as in the
# original.
df = pd.read_csv(f"{GOT_DIR}/datasets/sp_en_trans.csv")
true_stmts = [s for s in df[df.label == 1]["statement"].tolist() if s not in prompt]
false_stmts = [s for s in df[df.label == 0]["statement"].tolist() if s not in prompt]


def pd_diff(statements, mode):
    """Mean P(TRUE) - P(FALSE) over `statements`, applying the mass-mean direction
    per `mode` in {'none','add','subtract'} at the two tokens around the period,
    across the intervention band (interventions.py's probability-difference PD)."""
    queries = [prompt + s + " This statement is:" for s in statements]

    def fn(act, node):
        if mode == "none":
            return act
        act = act.clone()
        for offset in (-1, 0):  # the token before the period, and the period
            act[:, -LEN_SUFFIX + offset, :] += (direction if mode == "add" else -direction)
        return act

    outs = []
    for i in range(0, len(queries), BATCH):
        toks = model.tokenizer(queries[i:i + BATCH], return_tensors="pt",
                               padding=True, return_token_type_ids=False)
        logits = model.forward_logits(toks, fn=fn, layers=INTERVENE_LAYERS, modules="residual")
        probs = logits[:, -1, :].float().softmax(-1)
        outs.append((probs[:, true_id] - probs[:, false_id]).detach().cpu())
    return torch.cat(outs).mean().item()


# The paper's four probability-difference terms (Sec. 6.1):
PD_plus = pd_diff(true_stmts, "none")            # PD+  : true statements,  no intervention
PD_minus = pd_diff(false_stmts, "none")          # PD-  : false statements, no intervention
PD_star_minus = pd_diff(false_stmts, "add")      # PD*- : false + dir  (push false -> true)
PD_star_plus = pd_diff(true_stmts, "subtract")   # PD*+ : true  - dir  (push true  -> false)

# Normalized Indirect Effect: the fraction of the natural true/false gap the
# intervention closes (NIE=1: flips the label as confidently as a genuine one).
NIE_ft = (PD_star_minus - PD_minus) / (PD_plus - PD_minus)   # false -> true  (add)
NIE_tf = (PD_star_plus - PD_plus) / (PD_minus - PD_plus)      # true  -> false (subtract)

print("intervene layers:", INTERVENE_LAYERS,
      f"| {len(true_stmts)} true / {len(false_stmts)} false eval statements")
print(f"PD+  (true,  none)  = {PD_plus:+.3f}")
print(f"PD-  (false, none)  = {PD_minus:+.3f}")
print(f"PD*- (false, +dir)  = {PD_star_minus:+.3f}   ->  NIE(false->true) = {NIE_ft:.2f}")
print(f"PD*+ (true,  -dir)  = {PD_star_plus:+.3f}   ->  NIE(true->false) = {NIE_tf:.2f}")

intervene layers: [8, 9, 10, 11, 12, 13, 14] | 174 true / 176 false eval statements
PD+  (true,  none)  = +0.783
PD-  (false, none)  = -0.914
PD*- (false, +dir)  = +0.582   ->  NIE(false->true) = 0.88
PD*+ (true,  -dir)  = -0.876   ->  NIE(true->false) = 0.98


In [10]:
# LaTeX table: paper's 13B NIE vs our reproduced 13B NIE (same metric).
# -> tables/intervention.txt
latex = rf"""\begin{{table}}[tbp]
  \centering\small
  \begin{{tabular}}{{lcc}}
    \hline
     & \textbf{{\citet{{marks2023geometry}}}}, NIE & \textbf{{Murano}}, NIE \\
    Intervention (MM dir., train cities+neg\_cities) & 13B & 13B \\
    \hline
    subtract ($-$dir, true$\to$false) & .97 & {NIE_tf:.2f} \\
    add ($+$dir, false$\to$true)      & .85 & {NIE_ft:.2f} \\
    \hline
  \end{{tabular}}
  \caption{{Normalized indirect effect (NIE) of the mass-mean truth direction on
  \texttt{{sp\_en\_trans}} at layers {INTERVENE_LAYERS[0]}--{INTERVENE_LAYERS[-1]} of
  LLaMA-2-13B, reproducing \citet{{marks2023geometry}} Table~2 with the paper's exact
  metric. An NIE of 1 means the intervention flips the model's judgement as
  confidently as a genuine label; 0 means no effect. Baseline probability differences
  $PD^{{+}}={PD_plus:.3f}$ (true), $PD^{{-}}={PD_minus:.3f}$ (false); intervened
  $PD^{{*-}}={PD_star_minus:.3f}$ (false$+$dir), $PD^{{*+}}={PD_star_plus:.3f}$
  (true$-$dir). Same model and same metric as the paper's 13B column.}}
  \label{{tab:got-intervention}}
\end{{table}}
"""
write_latex(latex, "intervention.txt")

saved tables/intervention.txt 

\begin{table}[tbp]
  \centering\small
  \begin{tabular}{lcc}
    \hline
     & \textbf{\citet{marks2023geometry}}, NIE & \textbf{Murano}, NIE \\
    Intervention (MM dir., train cities+neg\_cities) & 13B & 13B \\
    \hline
    subtract ($-$dir, true$\to$false) & .97 & 0.98 \\
    add ($+$dir, false$\to$true)      & .85 & 0.88 \\
    \hline
  \end{tabular}
  \caption{Normalized indirect effect (NIE) of the mass-mean truth direction on
  \texttt{sp\_en\_trans} at layers 8--14 of
  LLaMA-2-13B, reproducing \citet{marks2023geometry} Table~2 with the paper's exact
  metric. An NIE of 1 means the intervention flips the model's judgement as
  confidently as a genuine label; 0 means no effect. Baseline probability differences
  $PD^{+}=0.783$ (true), $PD^{-}=-0.914$ (false); intervened
  $PD^{*-}=0.582$ (false$+$dir), $PD^{*+}=-0.876$
  (true$-$dir). Same model and same metric as the paper's 13B column.}
  \label{tab:got-intervention}
\end{table}



## 7. Recap and comparison to the paper

You rebuilt the geometry-of-truth analysis end to end with murano's primitives:
`Record` for the last-token residual reads, the mass-mean and logistic probes for
decoding, `SteeringVector` for the truth direction, and `forward_logits` for the
causal intervention. The findings match Marks & Tegmark:

| What | Marks & Tegmark | This notebook |
|---|---|---|
| Truth is linear | true/false separate in the top PCs | clear split along a principal component (§3) |
| Negation is separate | affirmative/negated directions near-orthogonal | cities vs. neg_cities near-orthogonal (§3) |
| Structure emerges with depth | split forms in the early-middle layers | split sharpens across layers (§4) |
| One direction generalizes | probes transfer across topics | LR/MM transfer, negated sets anti-correlate (§5) |
| The direction is causal | adding/subtracting it flips the judgement | NIE ≈ 0.9 both directions (§6) |

The method is the paper's: last-token residual reads, PCA via SVD, the mass-mean
and logistic probes with the whitened decision rule, and the unscaled mass-mean
direction applied over an intervention band, scored by the normalized indirect
effect.

**Where to go next.** Sweep the probe layer to trace where truth is most linearly
readable; add more of the paper's datasets to the generalization grid; or vary the
intervention band and direction scale to map where and how strongly the truth
direction acts.

## Citation

```bibtex
@article{marks2023geometry,
  author  = {Samuel Marks and Max Tegmark},
  title   = {The Geometry of Truth: Emergent Linear Structure in Large Language
             Model Representations of True/False Datasets},
  journal = {CoRR}, volume = {abs/2310.06824}, year = {2023},
  url     = {https://doi.org/10.48550/arXiv.2310.06824}
}
```